# Figure S6a — Example meshes coloured by synapse density

Same geodesic density calculation as Figure 3h,i (R = 4 µm Dijkstra neighbourhood on the
mesh vertex graph), but rendered with two separate single-hue scales instead of a shared
viridis one, so the two synapse types can be told apart at a glance:

| Map | Synapses | Colour ramp |
| --- | --- | --- |
| **axoaxonic** (top in the figure) | sensory neuron postsynaptic | PuRd → `#cb2f43` (red) |
| **axodendritic** (bottom) | sensory neuron presynaptic | PuBu → `#1c4286` (blue) |

**Output.** Colour-baked `.ply` meshes and the two colourbar SVGs.

**Requires CAVE** — mesh fetched through CloudVolume, synapses queried live.


In [ ]:
# Quick visual test of custom 2-color maps (no saving)
# PRE:  PuBu start  -> #1c4286 (bluer, more saturated)
# POST: PuRd start  -> #cb2f43 (redder, more saturated)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap

# --- controls ---
START_FRAC_PRE  = 0.00  # where to sample the "start" from PuBu (0.0 = very first color)
START_FRAC_POST = 0.00  # where to sample the "start" from PuRd
END_HEX_PRE  = "#006eb8"  # saturated blue
END_HEX_POST = "#cb2f43"  # saturated red

# --- build the two 2-color colormaps ---
start_pre  = cm.get_cmap("Blues")(START_FRAC_PRE)
start_post = cm.get_cmap("Reds")(START_FRAC_POST)
end_pre    = mcolors.to_rgba(END_HEX_PRE)
end_post   = mcolors.to_rgba(END_HEX_POST)

cmap_pre_custom  = LinearSegmentedColormap.from_list("PuBu_to_blue", [start_pre,  end_pre])
cmap_post_custom = LinearSegmentedColormap.from_list("PuRd_to_red",  [start_post, end_post])

# --- make simple gradient strips ---
x = np.linspace(0, 1, 512)
grad = np.tile(x, (40, 1))  # 40 px tall strip

fig, axes = plt.subplots(2, 1, figsize=(8, 2.4), constrained_layout=True)

axes[0].imshow(grad, aspect="auto", cmap=cmap_pre_custom)
axes[0].set_axis_off()
axes[0].set_title("PRE: PuBu → #1c4286", fontsize=10)

axes[1].imshow(grad, aspect="auto", cmap=cmap_post_custom)
axes[1].set_axis_off()
axes[1].set_title("POST: PuRd → #cb2f43", fontsize=10)

plt.show()


In [ ]:
# ===============================================
# Synapse-density coloring (geodesic, surface R)
# - Builds pre/post densities for each neuron
# - Clipping for stronger contrast: PRE≤30, POST≤10
# - Colors:
#     PRE  → PuBu(start) → #1c4286 (custom blue)
#     POST → PuRd(start) → #cb2f43 (custom red)
# - Exports PLYs and two SVG colorbars
# ===============================================

import os
from typing import Optional, Dict, Tuple
import numpy as np
from caveclient import CAVEclient
import cloudvolume as cv
import trimesh
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import Normalize, LinearSegmentedColormap

# -----------------------------
# Configuration (edit as needed)
# -----------------------------
root_ids = [
   720575940922958792, # cltmr
   720575940920833705, # mrgd
   720575940893392149, # ad-ltmr
   720575940881926778, # ab-ltmr
   720575940882155492, # sst
   720575940847552365, # ad-htmr
   720575940881918801, # chtmr-pep
   720575940927175960, # cold
]

DATASTACK = "wclee_mouse_spinalcord_cltmr"
SEG_URL = "graphene://https://cave.fanc-fly.com/segmentation/table/wclee_mouse_spinalcord_cltmr"

# If your CloudVolume requires a secrets token, set it here (or to None if not needed)
CV_SECRETS: Optional[str] = None  # e.g., "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

# Radius for geodesic neighborhood (mesh units; nm is typical)
R: float = 4000.0

# Synapse table + materialization version
SYN_TABLE = "sensory_neuron_synapses"

# Scale synapse centroids from voxel → nm if materialize returns vox coords
RES_NM_PER_VOX = np.array([16.0, 16.0, 45.0], dtype=float)

# Output directory
OUTPUT_DIR = "figureS6a_mesh_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------- Custom colormap controls (tweak here) -------
# Start color is sampled from PuBu/PuRd at this fraction.
# 0.00 = very first color (pale). Try 0.08–0.12 for deeper starts.
START_FRAC_PRE  = 0.00   # for PuBu (pre / blue)
START_FRAC_POST = 0.00   # for PuRd (post / red)

# Target END colors (more saturated)
END_HEX_PRE  = "#006eb8"  # saturated blue
END_HEX_POST = "#cb2f43"  # saturated red

# Clipping caps to emphasize contrast
VMAX_PRE  = 30.0  # pre densities ≥30 map to the end color
VMAX_POST = 10.0  # post densities ≥10 map to the end color

# -----------------------------
# Helpers
# -----------------------------
def get_scaled_synapses(client: CAVEclient,
                        seg_id: int,
                        query_type: str,
                        syn_table: str,
                        mat_version: int,
                        res_nm_per_vox: np.ndarray) -> np.ndarray:
    """
    query_type: 'pre' or 'post'
    Returns Nx3 array of synapse centroid positions in nm.
    """
    df = client.materialize.synapse_query(
        **{f"{query_type}_ids": [seg_id]},
        synapse_table=syn_table,
        materialization_version=mat_version,
        split_positions=True
    )
    if df.empty:
        return np.empty((0, 3), dtype=float)

    pts = np.stack([
        np.hstack(df['ctr_pt_position_x'].values),
        np.hstack(df['ctr_pt_position_y'].values),
        np.hstack(df['ctr_pt_position_z'].values)
    ], axis=1).astype(float)
    return pts * res_nm_per_vox  # scale to nm

def build_vertex_graph_with_weights(t_mesh: trimesh.Trimesh) -> nx.Graph:
    """NetworkX graph with 'weight' = Euclidean edge length."""
    G = t_mesh.vertex_adjacency_graph
    verts = t_mesh.vertices
    for u, v in G.edges:
        G[u][v]['weight'] = float(np.linalg.norm(verts[u] - verts[v]))
    return G

def density_from_synapses_on_surface(t_mesh: trimesh.Trimesh,
                                     syn_pts_nm: np.ndarray,
                                     G: nx.Graph,
                                     radius_nm: float) -> np.ndarray:
    """
    For each synapse point:
      - snap to nearest surface vertex
      - Dijkstra with cutoff 'radius_nm'
      - increment every vertex reached
    """
    n_verts = t_mesh.vertices.shape[0]
    if syn_pts_nm.shape[0] == 0:
        return np.zeros(n_verts, dtype=float)

    ProxQuery = trimesh.proximity.ProximityQuery(t_mesh)
    _, closest_vertices = ProxQuery.vertex(syn_pts_nm)  # (dists, idx)

    density = np.zeros(n_verts, dtype=float)
    unique_idx, counts = np.unique(closest_vertices, return_counts=True)

    for syn_vtx, mult in zip(unique_idx, counts):
        g_dists = nx.single_source_dijkstra_path_length(
            G, int(syn_vtx), cutoff=radius_nm, weight='weight'
        )
        for v in g_dists.keys():
            density[v] += float(mult)
    return density

def _safe_concat(dct: Dict[int, np.ndarray]) -> np.ndarray:
    arrs = []
    for a in dct.values():
        a = np.asarray(a, dtype=float)
        a = np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)
        arrs.append(a)
    if len(arrs) == 0:
        return np.array([], dtype=float)
    return np.concatenate(arrs)

def save_colorbar_svg(cmap, norm: Normalize, title: str, svg_path: str) -> None:
    """Saves a horizontal colorbar for the given colormap/norm as an SVG."""
    fig, ax = plt.subplots(figsize=(8, 1.2))
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=ax, orientation="horizontal")

    vmin, vmax = float(norm.vmin), float(norm.vmax)
    ticks = np.linspace(vmin, vmax, 5)
    tick_labels = [f"{t:.0f}" for t in ticks]

    cbar.set_ticks(ticks)
    cbar.set_ticklabels(tick_labels)
    cbar.set_label(title, fontsize=10)
    cbar.ax.tick_params(labelsize=8)

    plt.savefig(svg_path, format="svg", dpi=300, bbox_inches="tight")
    plt.close(fig)

def make_two_color_cmap(start_rgba, end_rgba, name):
    """Simple linear 2-color map from start to end RGBA."""
    return LinearSegmentedColormap.from_list(name, [start_rgba, end_rgba])

# -----------------------------
# Main build: meshes + densities
# -----------------------------
client = CAVEclient(DATASTACK)
# Pin the materialization so the panel is reproducible.
MATERIALIZATION = 669
mat_version = (MATERIALIZATION if MATERIALIZATION is not None
               else client.materialize.most_recent_version())

pre_density_all: Dict[int, np.ndarray] = {}
post_density_all: Dict[int, np.ndarray] = {}
mesh_all: Dict[Tuple[int, str], trimesh.Trimesh] = {}

for root_id in root_ids:
    # Resolve seg_id
    seg_id = client.chunkedgraph.get_latest_roots(root_id)[0]

    # Fetch mesh
    vol = cv.CloudVolume(SEG_URL, secrets=CV_SECRETS, use_https=True, progress=False)
    cv_mesh = vol.mesh.get(seg_id)
    if seg_id not in cv_mesh:
        print(f"WARNING: seg_id {seg_id} mesh not found; skipping")
        continue
    mesh_obj = cv_mesh[seg_id]

    base_mesh = trimesh.Trimesh(vertices=mesh_obj.vertices,
                                faces=mesh_obj.faces,
                                process=False)

    # Graph once per neuron
    G = build_vertex_graph_with_weights(base_mesh)

    # Synapses (nm)
    pre_pts_nm  = get_scaled_synapses(client, seg_id, "pre",  SYN_TABLE, mat_version, RES_NM_PER_VOX)
    post_pts_nm = get_scaled_synapses(client, seg_id, "post", SYN_TABLE, mat_version, RES_NM_PER_VOX)

    # Densities
    pre_density  = density_from_synapses_on_surface(base_mesh, pre_pts_nm,  G, R)
    post_density = density_from_synapses_on_surface(base_mesh, post_pts_nm, G, R)

    pre_density_all[root_id] = pre_density
    post_density_all[root_id] = post_density

    mesh_all[(root_id, 'pre')]  = base_mesh
    mesh_all[(root_id, 'post')] = base_mesh

# -----------------------------
# Clipped normalization + custom colormaps
# -----------------------------
# (maxima computed for reference only)
all_pre  = _safe_concat(pre_density_all)
all_post = _safe_concat(post_density_all)
max_pre_density  = float(all_pre.max())  if all_pre.size  else 0.0
max_post_density = float(all_post.max()) if all_post.size else 0.0

# Build 2-color maps:
# PRE (blue): PuBu start -> #1c4286
# POST (red): PuRd start -> #cb2f43
start_pre_rgba  = cm.get_cmap("Blues")(START_FRAC_PRE)
start_post_rgba = cm.get_cmap("Reds")(START_FRAC_POST)
end_pre_rgba    = mcolors.to_rgba(END_HEX_PRE)
end_post_rgba   = mcolors.to_rgba(END_HEX_POST)

cmap_pre  = make_two_color_cmap(start_pre_rgba,  end_pre_rgba,  "PRE_PuBu_to_customBlue")
cmap_post = make_two_color_cmap(start_post_rgba, end_post_rgba, "POST_PuRd_to_customRed")

# Fixed-range norms using your clip caps
norm_pre  = Normalize(vmin=0.0, vmax=VMAX_PRE)
norm_post = Normalize(vmin=0.0, vmax=VMAX_POST)

# -----------------------------
# Export colored meshes (PLY) with clipping
# -----------------------------
for root_id in root_ids:
    # PRE (blue) -- clip at 30
    if root_id in pre_density_all and (root_id, 'pre') in mesh_all:
        density = np.asarray(pre_density_all[root_id], dtype=float)
        density = np.nan_to_num(density, nan=0.0, posinf=0.0, neginf=0.0)
        density = np.clip(density, 0.0, VMAX_PRE)
        rgba = cmap_pre(norm_pre(density))
        vertex_colors = (rgba[:, :3] * 255).astype(np.uint8)

        mesh = mesh_all[(root_id, 'pre')]
        t_mesh = trimesh.Trimesh(vertices=mesh.vertices,
                                 faces=mesh.faces,
                                 vertex_colors=vertex_colors,
                                 process=False)
        t_mesh.visual.vertex_colors = vertex_colors

        out_path = os.path.join(OUTPUT_DIR, f"{root_id}_pre_customBlue.ply")
        with open(out_path, 'wb') as f:
            f.write(trimesh.exchange.ply.export_ply(t_mesh))

    # POST (red) -- clip at 10
    if root_id in post_density_all and (root_id, 'post') in mesh_all:
        density = np.asarray(post_density_all[root_id], dtype=float)
        density = np.nan_to_num(density, nan=0.0, posinf=0.0, neginf=0.0)
        density = np.clip(density, 0.0, VMAX_POST)
        rgba = cmap_post(norm_post(density))
        vertex_colors = (rgba[:, :3] * 255).astype(np.uint8)

        mesh = mesh_all[(root_id, 'post')]
        t_mesh = trimesh.Trimesh(vertices=mesh.vertices,
                                 faces=mesh.faces,
                                 vertex_colors=vertex_colors,
                                 process=False)
        t_mesh.visual.vertex_colors = vertex_colors

        out_path = os.path.join(OUTPUT_DIR, f"{root_id}_post_customRed.ply")
        with open(out_path, 'wb') as f:
            f.write(trimesh.exchange.ply.export_ply(t_mesh))

# -----------------------------
# Export two SVG colorbars (showing clipped ranges)
# -----------------------------
save_colorbar_svg(
    cmap_pre,
    norm_pre,
    title="Pre synapse density (clipped at 30)",
    svg_path=os.path.join(OUTPUT_DIR, "colormap_PRE_PuBu_to_1c4286.svg"),
)
save_colorbar_svg(
    cmap_post,
    norm_post,
    title="Post synapse density (clipped at 10)",
    svg_path=os.path.join(OUTPUT_DIR, "colormap_POST_PuRd_to_cb2f43.svg"),
)

print("Done.")
print("Output dir:", OUTPUT_DIR)
print("max_pre_density (observed):", max_pre_density)
print("max_post_density (observed):", max_post_density)
print("VMAX_PRE used:", VMAX_PRE, "VMAX_POST used:", VMAX_POST)


In [ ]:
# Save tiny SVG colorbars only (no meshes)
# PRE (blue):  PuBu(start) → #006eb8, range 0–30
# POST (red):  PuRd(start) → #cb2f43, range 0–10

import os
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap, Normalize

OUT_DIR = "figureS6a_mesh_output"
os.makedirs(OUT_DIR, exist_ok=True)

# Appearance: Helvetica, keep text as text in SVG
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "svg.fonttype": "none",
})

# Controls
START_FRAC_PRE  = 0.00   # sample start from PuBu (0.0 = first color)
START_FRAC_POST = 0.00   # sample start from PuRd
END_HEX_PRE  = "#006eb8"  # saturated blue
END_HEX_POST = "#cb2f43"  # saturated red
VMAX_PRE  = 30.0
VMAX_POST = 10.0

def build_two_color_cmap(start_name, start_frac, end_hex, name):
    start_rgba = cm.get_cmap(start_name)(start_frac)
    end_rgba   = mcolors.to_rgba(end_hex)
    return LinearSegmentedColormap.from_list(name, [start_rgba, end_rgba])

def save_colorbar_svg(cmap, vmin, vmax, label, out_path):
    # 2 cm x 0.5 cm
    inch_w, inch_h = 2/2.54, 0.5/2.54
    fig = plt.figure(figsize=(inch_w, inch_h))
    cax = fig.add_axes([0.05, 0.40, 0.90, 0.35])  # tight, horizontal strip

    norm = Normalize(vmin=vmin, vmax=vmax)
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    cbar = plt.colorbar(sm, cax=cax, orientation="horizontal")
    cbar.outline.set_visible(False)  # no black outline

    # Minimal ticks (start, mid, end)
    ticks = [vmin, 0.5*(vmin+vmax), vmax]
    cbar.set_ticks(ticks)
    cbar.set_ticklabels([f"{t:.0f}" for t in ticks])
    cbar.ax.tick_params(labelsize=4, length=2, width=0.4)

    cbar.set_label(label, fontsize=4, labelpad=1)

    # Hide spines just in case
    for spine in cbar.ax.spines.values():
        spine.set_visible(False)

    fig.savefig(out_path, format="svg", bbox_inches="tight", pad_inches=0)
    plt.close(fig)

# Build maps
cmap_pre  = build_two_color_cmap("PuBu", START_FRAC_PRE,  END_HEX_PRE,  "pre_blue")
cmap_post = build_two_color_cmap("PuRd", START_FRAC_POST, END_HEX_POST, "post_red")

# Save SVGs
save_colorbar_svg(cmap_pre,  0.0, VMAX_PRE,  "Axodendritic synapse density",  os.path.join(OUT_DIR, "colormap_pre_blue.svg"))
save_colorbar_svg(cmap_post, 0.0, VMAX_POST, "Axoaxonic synapse density", os.path.join(OUT_DIR, "colormap_post_red.svg"))

print("Saved to:", OUT_DIR)
